In [2]:
import numpy as np
import pandas as pd
import joblib
from keras.models import load_model
from sklearn.metrics import mean_absolute_error

SEQ_LEN = 1

MODEL_PATH = "energy_model_final3.keras"
SCALER_X_PATH = "scaler_x.pkl"
SCALER_Y_PATH = "scaler_y.pkl"

INPUT_CSV = "consumptions.csv"



WEEKDAY_MAP = {
    "Monday":0,"Tuesday":1,"Wednesday":2,
    "Thursday":3,"Friday":4,"Saturday":5,"Sunday":6
}



# Load model and scalers

model = load_model(MODEL_PATH)
scaler_x = joblib.load(SCALER_X_PATH)
scaler_y = joblib.load(SCALER_Y_PATH)

FEATURES = list(scaler_x.feature_names_in_)
CITY_COLS = [c for c in FEATURES if c.startswith("city_")]
DOW_COLS  = [c for c in FEATURES if c.startswith("dow_")]



# Feature builder

def build_features(df):

    df = df.copy()

    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

    
    df = df.sort_values(["city","Timestamp"]).reset_index(drop=True)

    
    city_dummies = pd.get_dummies(df["city"], prefix="city")
    df = pd.concat([df, city_dummies], axis=1)

    for c in CITY_COLS:
        if c not in df:
            df[c] = 0

    # weekday encoding
    df["weekday_num"] = df["weekday"].map(WEEKDAY_MAP)

    dow_dummies = pd.get_dummies(df["weekday_num"], prefix="dow")
    df = pd.concat([df, dow_dummies], axis=1)

    for c in DOW_COLS:
        if c not in df:
            df[c] = 0

    # time features
    df["year"]      = df["Timestamp"].dt.year
    df["month"]     = df["Timestamp"].dt.month
    df["dayofyear"] = df["Timestamp"].dt.dayofyear

    df["hour_sin"]  = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"]  = np.cos(2*np.pi*df["hour"]/24)

    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

    hour_week = df["hour"] + df["weekday_num"]*24
    df["hour_week_sin"] = np.sin(2*np.pi*hour_week/168)
    df["hour_week_cos"] = np.cos(2*np.pi*hour_week/168)

    # weather interactions
    df["temp_sq"]       = df["tavg"]**2
    df["temp_humidity"] = df["tavg"] * df["humidity"]
    df["temp_wind"]     = df["tavg"] * df["wspd"]
    df["wind_humidity"] = df["wspd"] * df["humidity"]

    
    X = df.reindex(columns=FEATURES, fill_value=0)

    X = scaler_x.transform(X).astype(np.float32)

    return X.reshape(len(X), SEQ_LEN, len(FEATURES)), df



# MAIN

df = pd.read_csv(INPUT_CSV)

X, df_sorted = build_features(df)

pred_scaled = model.predict(X, verbose=1)

pred_log = scaler_y.inverse_transform(pred_scaled)

preds = np.expm1(pred_log).flatten()

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = df_sorted["EnergyConsumption"].values

# --- Core metrics ---
mae = mean_absolute_error(y_true, preds)
rmse = np.sqrt(mean_squared_error(y_true, preds))
mape = np.mean(np.abs((y_true - preds) / (y_true + 1e-8))) * 100
r2 = r2_score(y_true, preds)

# --- Additional diagnostics ---
bias = np.mean(preds - y_true)

# peak error = error at max true demand (important in energy forecasting)
peak_idx = np.argmax(y_true)
peak_error = preds[peak_idx] - y_true[peak_idx]

# --- Print ---
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("MAPE:", round(mape, 2), "%")
print("R2:", round(r2, 4))
print("Bias:", round(bias, 4))
print("Peak Error:", round(peak_error, 4))

263/263 [==============================] - 1s 1ms/step
MAE: 2182.478
RMSE: 3151.4035
MAPE: 5.82 %
R2: 0.7696
Bias: 161.9116
Peak Error: -1618.1427
